# Job 50536365 — the second SAE dictionary: the structure replicates, the exact peak layer does not

Findings 19, 21 and 29 all rest on **one dictionary per layer at one seed**, with `best_f1` a max
over 256 predicates. That caveat is written into all three and it is the open item that kept them
from being a paper figure. This run is the replication: same SOLVER checkpoint (finding 23 cell B,
the leak-free model that scores **1.0000** on the shared pool), same seven layers, same target L0,
every SAE hyper-parameter byte-identical to job `50075499` — **`--seed 1` is the only change**.

**Headline.** Three of the four pre-registrations survive, and the one that fails is specific and
useful:

* **The asymmetry is real.** `justifies_next` features at `pre_node`: **0 at all seven layers, in
  both seeds**. The forcing set is encoded only while the model picks the **colour**, which is when
  DD's elimination rule needs it. This is the claim findings 21(d) and 29(b) rest on and it now has
  an independent dictionary behind it.
* **The peak layer moves.** At `pre_color`, `best_justifier_f1` peaks at **L8 (0.768)** in seed 0 and
  at **L10 (0.839)** in seed 1, with the other of the pair close behind in each case (L10 0.623 /
  L8 0.790). So finding 29's "the forcing set enters at **L8**" must be quoted as **L8–L10**. It is
  still inside job `50075339`'s L6→L10 patching handoff, so the agreement between the correlational
  and causal localisations survives — at window resolution, not at layer resolution.
* **The forcing-set counts replicate; the layer-0 identity counts do not.** Total `justifies_next`
  at `pre_color` goes 47 → 53 (per layer: 2→2, 17→22, 28→27, 0→2). But layer 0's `next_node` count
  goes **23 → 56** at an unchanged best F1 of exactly **1.000**, and section 5 shows why: layer 0's
  dictionary never reaches its target sparsity in either seed.

**What this does not do.** `--seed` moves the instance pool (`seed_start = 600000000 + seed`), the
position sample, the SAE initialisation and the null-shift RNG together, so this is a *full
replication*, not an initialisation-only ablation. Agreement is therefore strong evidence that the
structure is in the model; the one disagreement cannot be attributed to any single source.


In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S = os.environ["SCRATCH"] + "/spin_gnn"
RUNS = {0: f"{S}/sudoku_sae_solved_2026-10-28/leakfree1000_l0_64",        # 50075499, finding 29
        1: f"{S}/sudoku_sae_solved_2026-10-31/leakfree1000_l0_64_seed1"}  # 50536365, this run
pd.set_option("display.width", 220)


def load(kind):
    f = "sae_summary_by_kind.csv" if kind else "sae_summary.csv"
    out = []
    for seed, d in RUNS.items():
        t = pd.read_csv(f"{d}/{f}")
        t["seed"] = seed
        out.append(t)
    t = pd.concat(out, ignore_index=True)
    t["jn"] = t.family_counts.map(lambda s: json.loads(s).get("justifies_next", 0))
    t["nn"] = t.family_counts.map(lambda s: json.loads(s).get("next_node", 0))
    return t


bykind, pooled = load(True), load(False)
LAYERS = sorted(bykind.layer.unique())
print("layers", LAYERS, "| seeds", sorted(bykind.seed.unique()))


## 1. Why the two runs are comparable

A replication only means something if the two dictionaries were fit under the same pressure. They
were, and not by assumption — the l1 search landed on the **identical `l1_final` at every layer**
(0.0, 0.1, 0.1, 0.1414, 0.2, 0.4, 0.2828), `frac_variance_explained` agrees to three or four
decimals, and achieved sparsity `l0` agrees to about 1 in 60. The one layer where **both** seeds
miss the target is **layer 0** (`hit_target_l0 = 0`, achieved `l0` ≈ 7–9 against 64), and that is
exactly where the one non-replicating number lives (section 5). Seed 1 also misses at **layer 4**,
but in the opposite direction and by far less (`l0` 74.2 against 64, 1341 features alive), which is
a tolerance overshoot rather than a collapsed dictionary — and layer 4's `justifies_next` count is
0 in both seeds either way.

The model is also untouched: `spin_gnn/sudoku_sae.py` has not changed since `10c38b0f`, which is the
commit findings 19/21/29 were measured under, so the new rows drop straight into their tables.


In [ ]:
cols = ["l1_final", "l0", "hit_target_l0", "frac_variance_explained", "n_alive", "n_dead_features"]
print("=== dictionary fit, seed 0 vs seed 1 (the comparability check)")
print(pooled.pivot_table(index="layer", columns="seed", values=cols).round(4).to_string())


## 2. The metrics table

Split by position kind, because that split *is* finding 21: `pre_node` is the position where the
model chooses **which node** to emit, `pre_color` where it chooses **what colour** that node takes.
`justifies_next[u]` means "u is a KNOWN neighbour of the node being emitted" — exactly the input to
DD's elimination rule — and `best_justifier_f1` is the best single feature for any such predicate.


In [ ]:
for kind in ("pre_color", "pre_node"):
    t = bykind[bykind.position_kind == kind]
    piv = t.pivot_table(index="layer", columns="seed",
                        values=["jn", "best_justifier_f1", "nn", "best_next_node_f1",
                                "n_alive", "median_best_f1_null"])
    print(f"=== {kind}   (jn = justifies_next feature count, nn = next_node feature count)")
    print(piv.round(3).to_string(), "\n")


## 3. The four pre-registrations, scored

Written into `scripts/slurm/sudoku2_sae_solved_seed1.sbatch` **before submission**, against seed 0's
exact table.

| | claim | verdict |
|---|---|---|
| (a) | zero `justifies_next` at `pre_node`, all seven layers | **CONFIRMED** |
| (b) | layer 0: `next_node` F1 ≥ 0.99 and 18–28 features at `pre_color`; 0 and below-null at `pre_node` | **SPLIT** — F1 and the asymmetry hold, the count does not (23 → 56) |
| (c) | `best_justifier_f1` at L8 ≥ 0.70 **and** L8 is the argmax | **SPLIT** — 0.790 ≥ 0.70 holds, the argmax moves to L10 |
| (d) | total `justifies_next` at `pre_color` within ±40% of 47 | **CONFIRMED** (53) |

(c) is the one that changes a written claim. Finding 29 says the forcing set "enters at L8", and
page 14 of the verdict deck reads the DD steps off that layer index. Two seeds put the peak one rung
apart, so the honest statement is **L8–L10**, a window, and the thing that survives is that the
forcing-set representation is concentrated in the back half and nowhere near layer 0 or layer 2.


In [ ]:
pc = bykind[bykind.position_kind == "pre_color"].set_index(["seed", "layer"])
pn = bykind[bykind.position_kind == "pre_node"].set_index(["seed", "layer"])

print("(a) justifies_next at pre_node, by seed:",
      {s: [int(pn.loc[(s, L), "jn"]) for L in LAYERS] for s in (0, 1)},
      "-> CONFIRMED" if (pn.jn == 0).all() else "-> FALSIFIED")

for s in (0, 1):
    print(f"(b) seed {s}: layer 0 pre_color next_node F1 {pc.loc[(s, 0), 'best_next_node_f1']:.3f}, "
          f"count {int(pc.loc[(s, 0), 'nn'])}; pre_node F1 {pn.loc[(s, 0), 'best_next_node_f1']:.3f} "
          f"vs its null {pn.loc[(s, 0), 'median_best_f1_null']:.3f}, count {int(pn.loc[(s, 0), 'nn'])}")

for s in (0, 1):
    f = pc.loc[s, "best_justifier_f1"]
    print(f"(c) seed {s}: argmax layer {int(f.idxmax())} at {f.max():.3f}; "
          f"L8 {f.loc[8]:.3f}, L10 {f.loc[10]:.3f}")

for s in (0, 1):
    print(f"(d) seed {s}: total justifies_next at pre_color = {int(pc.loc[s, 'jn'].sum())}")


## 4. Comparison to the previous measurement on the same eval

There is no "previous best" here in the solve-rate sense — this is a replication, and the right
comparison is **seed 0 against seed 1 on the same checkpoint**, plus where that spread sits next to
the between-arm contrasts findings 19/21/29 are built on. Those arms were each measured at seed 0
only, so the table below mixes one replicated rung with two unreplicated ones and says so.

| arm | solve_rate | `justifies_next` @ `pre_color` | best F1 (layer) |
|---|---|---|---|
| leaked (finding 19/21) | 0.9960 | 15 | 0.247 (L8) |
| leak-free twin (finding 19/21) | 0.8540 | 69 | 0.743 (L8) |
| **SOLVER, seed 0** (finding 29) | **1.0000** | 47 | 0.768 (L8) |
| **SOLVER, seed 1** (this run) | **1.0000** | 53 | **0.839 (L10)**, L8 0.790 |

**The number that matters: the seed-to-seed spread on the SOLVER rung is 0.768 vs 0.790 at L8 (0.022)
and 47 vs 53 in count, against a leak-free-minus-leaked gap of 0.743 − 0.247 = 0.496.** The contrast
findings 19/21 rest on is more than twenty times the seed noise at the same layer, so a single seed
was enough for *that* claim even though it was not enough for the peak-layer claim.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
STY = {0: ("#2b6cb0", "o", "seed 0 (50075499)"), 1: ("#c53030", "s", "seed 1 (50536365)")}

for s, (c, mk, lab) in STY.items():
    r = pc.loc[s].sort_index()
    axes[0].plot(r.index, r.best_justifier_f1, mk + "-", color=c, lw=2.5, ms=9, label=lab,
                 rasterized=True)
    q = pn.loc[s].sort_index()
    axes[0].plot(q.index, q.best_justifier_f1, mk + "--", color=c, lw=1.6, ms=6, alpha=0.65,
                 label=lab + ", pre_node", rasterized=True)
    axes[1].bar(np.asarray(r.index) + (s - 0.5) * 0.8, r.jn, width=0.8, color=c, label=lab,
                rasterized=True)
axes[0].axvspan(6, 10, color="0.85", zorder=0)
axes[0].text(8, 0.04, "L6-L10: the patching handoff\n(job 50075339)", ha="center", fontsize=9)
axes[0].set_xlabel("layer"); axes[0].set_ylabel("best justifies_next F1")
axes[0].set_title("The forcing set: solid = pre_color, dashed = pre_node")
axes[0].legend(fontsize=9, loc="upper left")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("justifies_next features")
axes[1].set_title("Feature counts at pre_color")
axes[1].legend(fontsize=9)
for ax in axes:
    ax.set_xticks(LAYERS)
fig.tight_layout()


## 5. Where it is *not* robust, and the mechanism

Layer 0's `next_node` count goes **23 → 56** while its best F1 stays at exactly **1.000** in both
seeds. That is not a disagreement about what layer 0 represents; it is a disagreement about how many
copies of it the dictionary makes, and the fit table says why: **layer 0 is the one layer where
both seeds collapse well under the target sparsity** (`hit_target_l0 = 0`, achieved `l0` ≈ 7–9
against a target of 64), with about 3,950 of 4,096 features dead. The layer-0 residual stream is
close to
rank-deficient — `frac_variance_explained` is 1.0000 — so how many alive features survive the
`alive_rate` gate is close to arbitrary, and the count moved with it (99 → 145 alive).

**Consequence for the figure:** quote `best_f1` and the `pre_node`/`pre_color` asymmetry, which both
replicate; do not quote layer-0 feature *counts*. The `justifies_next` counts in the back half are
fine (2→2, 17→22, 28→27), which is why pre-registration (d) passed and (b)'s count clause failed —
the instability is specific to the degenerate layer, not to counting in general.


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
w = 0.38
for s, (c, _, lab) in STY.items():
    r = pooled[pooled.seed == s].sort_values("layer")
    ax.bar(np.arange(len(LAYERS)) + (s - 0.5) * w, r.n_alive, width=w, color=c, label=lab,
           rasterized=True)
    for i, (L, a, hit) in enumerate(zip(r.layer, r.n_alive, r.hit_target_l0)):
        if not hit:
            ax.text(i + (s - 0.5) * w, a + 60, "target\nmissed", ha="center", fontsize=8)
ax.set_xticks(np.arange(len(LAYERS)), [f"L{L}" for L in LAYERS])
ax.set_ylabel("alive features (of 4096)")
ax.set_title("Layer 0's dictionary is degenerate in both seeds - which is where the counts moved")
ax.legend()
fig.tight_layout()


## 6. Provenance, config, limitations

* **Job** `50536365`, submitted 2026-10-31 13:16:33, started 13:17:18, **COMPLETED 13:47:49**,
  elapsed **00:30:31**, one H100 on `sh04-02n01`. (Seed 0's job `50075499` took 01:34:07 for the
  same work on a different node; the difference is hardware, not configuration — `l1_final` and
  `frac_variance_explained` match layer by layer.)
* **Commit at submit** `54a0a1d`; the script committed as `31cbdc2`. The analysis module
  `spin_gnn/sudoku_sae.py` is unchanged since **`10c38b0f`**, the commit findings 19/21/29 ran under.
* **Config, byte-identical to `50075499` except `--seed`:** `--expansion 16 --l1-coeff 0.1
  --l1-max-tries 8 --sae-epochs 1500 --n-instances 3000 --max-positions-per-instance 40
  --alive-rate 1e-3 --n-null-shifts 19`, `--layer 0 2 4 6 8 10 12 --target-l0 64 --split-by-kind`.
  **Seed 1** (seed 0 is the run being replicated).
* **Checkpoint** `$SCRATCH/spin_gnn/sudoku_n24_data_2026-10-26/data240k_ep100/`
  `d0256_h04_L12_ff01024_lr1e-03_pre_shuf` — N=24, leak-free (`shuffle_adj=True`), 240000 × 100ep,
  shared-pool solve_rate 1.0000.
* **Outputs** `$SCRATCH/spin_gnn/sudoku_sae_solved_2026-10-31/leakfree1000_l0_64_seed1/`
  (`sae_summary.csv`, `sae_summary_by_kind.csv`, 21 per-layer feature CSVs). On `$SCRATCH`, so
  subject to the 90-day purge; the summaries are small and the dictionaries are re-derivable from
  the checkpoint while it lasts.

**Limitations.** Two seeds is two, not a distribution: the L8/L10 peak swap says the layer index is
uncertain, it does not say the uncertainty is exactly one rung. `best_f1` is still a max over 256
predicates in both seeds, so the multiple-comparisons ceiling is unchanged — the replication shows
the *same* ceiling produces the same answer twice, which is weaker than removing it. Only the SOLVER
arm is replicated; the leaked and 0.854 arms remain single-seed, so the between-arm contrasts in the
section-4 table still carry finding 19's original caveat. And the SAE remains correlational: what is
*used* comes from the interventions (findings 27, 28), not from here.

**Next.** If findings 19/21/29 become a paper figure, (i) print best F1 and the asymmetry, not
layer-0 counts, (ii) write the localisation as **L8–L10**, and (iii) the cheapest remaining
robustness buy is a second seed on the *leaked* arm, since that is the arm the headline contrast is
measured against.
